In [ ]:
## Pothan Tang, 8/15/25
## Calculates the electric field/force due to the DC and RF pseudopotential

import constants as c
import math
import numpy as np
import matplotlib.pyplot as plt

## Parameters: adjust these values to simulate the dark matter or the ion
Z_charge = 1; 
m = c.m;

## Constants
e = c.e
um = c.um
omega = c.omega
vk = c.vk
vrf = c.vrf
z0 = c.z0
xy1k = c.xy1k
xy2k = c.xy2k
pi = math.pi

## Region of interest
res = 1*c.um; # resolution of potential 
x_max = 100*c.um;
y_max = 100*c.um;
z_max = 200*c.um;
x = np.linspace(-1*x_max,x_max, int(2*x_max/res+1), endpoint=True); # -1cm<=x<=1cm 
y = np.linspace(-1*y_max,y_max, int(2*y_max/res+1), endpoint=True);  # -1cm<=y<=1cm
z = np.linspace(0,z_max, int(z_max/res+1), endpoint=True);  # 0cm<=z<=1cm

## Zoomed potential files, -100<x<100um,-100<y<100um,0<z<200um
original_shape_3d = (201,201,201)
phi_dc_flat = np.loadtxt('dc_potential_zoomed')
phi_rf_flat = np.loadtxt('rf_potential_max_zoomed')
pseudo_flat = np.loadtxt('pseudo_potential_zoomed')
phi_dc = phi_dc_flat.reshape(original_shape_3d)
phi_rf = phi_rf_flat.reshape(original_shape_3d)
pseudo = pseudo_flat.reshape(original_shape_3d)
energy_tot = e*phi_dc+pseudo

### DC Force

In [ ]:
### DC electric force
## Numerical calculation
fdcx,fdcy,fdcz = -(Z_charge*e) * np.array(np.gradient(phi_dc,res,res,res))
"""
# Plot the 3D vector field
X, Y, Z = np.meshgrid(x, y, z, indexing='ij')
X_flat, Y_flat, Z_flat = X.flatten(), Y.flatten(), Z.flatten()
fdcx_flat, fdcy_flat, fdcz_flat = fdcx.flatten(), fdcy.flatten(), fdcz.flatten()

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.quiver(X_flat, Y_flat, Z_flat, fdcx_flat, fdcy_flat, fdcz_flat, length=0.1)
plt.show()
"""

## Analytical calculation
# Functions
def divatan(up,down):
    #This is d(arctan2(up,down))/ddown up to a minus sign. It's useful for the pseudo-potential
    return up/(up**2+down**2)
def divatanup(up,down):
    #This is d(divatan(up,down))/dup
    return (down**2-up**2)/(up**2+down**2)**2
def divatandown(up,down):
    #This is d(divatan(up,down))/ddown
    return -2*up*down/(up**2+down**2)**2

def anatangrad(xi,yi,x,y,z,v): # gradient term of DC potential
    dy=y-yi
    dx=x-xi
    r = math.sqrt(dx**2+dy**2+z**2); # added distance
    dry2=z**2+dy**2
    drx2=z**2+dx**2
    divy=z*dx/(r*dry2); # divide by factor r
    divz=-dy*dx*(1/dry2+1/drx2)/r; # divide by factor r
    divx=z*dy/(r*drx2); # divide by factor r
    return (v/(2*math.pi))*np.array([divx,divy,divz])

def FDC_single(x1,y1,x2,y2,x,y,z,v):
    return -Z_charge*e * (anatangrad(x2,y2,x,y,z,v)-anatangrad(x2,y1,x,y,z,v)-anatangrad(x1,y2,x,y,z,v)+anatangrad(x1,y1,x,y,z,v))

def FDC(x,y,z,m=c.m,q=c.e,omrf=c.omega,ymin=c.y11,yedge1=c.y21,yedge2=c.y12,ymax=c.y22):
    z_original = z
    force = np.zeros(3)
    for k in range (0,40):
        if(k!=19 and k!=39):
            z = z-z0
        (x1k,y1k) = xy1k[k]
        (x2k,y2k) = xy2k[k]
        force_single = FDC_single(x1k,y1k,x2k,y2k,x,y,z,vk[k])
        force += force_single
        z = z_original
    return force

## Obtain results of x,y,z axis
fdc2 = np.zeros((int(2*x_max/res+1),int(2*y_max/res+1),int(z_max/res+1),3))
for p in range(0,len(x)):
    xc = x[p]
    fdc2[p][100][70] += FDC(xc,0,70*um)
for q in range (0,len(y)):
    yc = y[q]
    fdc2[100][q][70] += FDC(0,yc,70*um)
for r in range (0,len(z)):
    zc = z[r]
    fdc2[100][100][r] += FDC(0,0,zc)

print("Numerical calculation results:")
print(f"x-component: {fdcx[:,100,70]}; {fdcx[100,:,70]}; {fdcx[100,100,:]} ")
print(f"y-component: {fdcy[:,100,70]}; {fdcy[100,:,70]}; {fdcy[100,100,:]} ")
print(f"z-component: {fdcz[:,100,70]}; {fdcz[100,:,70]}; {fdcz[100,100,:]} ")
print("\nAnalytical calculation results:")
print(fdc2[:,100,70])
print(fdc2[100,:,70])
print(fdc2[100,100,:])

Numerical calculation results:
x-component: [ 4.23450470e-16  4.24952507e-16  4.27796841e-16  4.30314541e-16
  4.32505608e-16  4.34377193e-16  4.35931683e-16  4.37173843e-16
  4.38108444e-16  4.38740253e-16  4.39076424e-16  4.39121723e-16
  4.38883305e-16  4.38370705e-16  4.37586308e-16  4.36534882e-16
  4.35229540e-16  4.33677435e-16  4.31882143e-16  4.29855585e-16
  4.27603722e-16  4.25132513e-16  4.22452688e-16  4.19572592e-16
  4.16499376e-16  4.13240194e-16  4.09803391e-16  4.06197309e-16
  4.02430296e-16  3.98508310e-16  3.94438505e-16  3.90230417e-16
  3.85892391e-16  3.81428003e-16  3.76845598e-16  3.72153521e-16
  3.67355347e-16  3.62459421e-16  3.57472897e-16  3.52400541e-16
  3.47247124e-16  3.42018604e-16  3.36722136e-16  3.31360102e-16
  3.25939655e-16  3.20464373e-16  3.14937830e-16  3.09365988e-16
  3.03752422e-16  2.98100710e-16  2.92412043e-16  2.86693573e-16
  2.80948877e-16  2.75176764e-16  2.69380808e-16  2.63568163e-16
  2.57736444e-16  2.51889229e-16  2.46031284e-

### RF Pseudo-Force

In [ ]:
### RF pseudo-force

## Numerical calculation
frfx,frfy,frfz = -1 * np.array(np.gradient(pseudo,res,res,res)); # multiply by Z^2 for dark matter if not already

## Analytical calculation
def FRF(x,y,z,m=c.m,q=c.e,omrf=c.omega,VRF=c.vrf,ymin=c.y11,yedge1=c.y21,yedge2=c.y12,ymax=c.y22):
    #this is the pseudo potential, which is Z^2*(Div[PhiRF]/cos(om*t))^2/(4m*omega^2)
    divypart=divatan(z,yedge2-y)-divatan(z,yedge1-y)+divatan(z,ymin-y)-divatan(z,ymax-y)
    divzpart=divatan(yedge2-y,z)-divatan(yedge1-y,z)+divatan(ymin-y,z)-divatan(ymax-y,z) 
    divyparty=-2*divypart*(divatandown(z,yedge2-y)-divatandown(z,yedge1-y)+divatandown(z,ymin-y)-divatandown(z,ymax-y)) 
    divypartz=2*divypart*(divatanup(z,yedge2-y)-divatanup(z,yedge1-y)+divatanup(z,ymin-y)-divatanup(z,ymax-y)) 
    divzparty=-2*divzpart*(divatanup(yedge2-y,z)-divatanup(yedge1-y,z)+divatanup(ymin-y,z)-divatanup(ymax-y,z)) 
    divzpartz=2*divzpart*(divatandown(yedge2-y,z)-divatandown(yedge1-y,z)+divatandown(ymin-y,z)-divatandown(ymax-y,z)) 
    return -1*((VRF*Z_charge*q)/(2*pi*np.sqrt(m)*omrf))**2*np.array([divypartz*0,divzparty+divyparty,divzpartz+divypartz])

## Obtain results for x,y,z axis
frf2 = np.zeros((int(2*x_max/res+1),int(2*y_max/res+1),int(z_max/res+1),3))
for p in range(0,len(x)):
    xc = x[p]
    frf2[p][100][70] += FRF(xc,0,70*um)
for q in range (0,len(y)):
    yc = y[q]
    frf2[100][q][70] += FRF(0,yc,70*um)
for r in range (0,len(z)):
    zc = z[r]
    frf2[100][100][r] += FRF(0,0,zc)

print("Numerical calculation results:")
print(f"x-component: {frfx[:,100,70]}; {frfx[100,:,70]}; {frfx[100,100,:]} ")
print(f"y-component: {frfy[:,100,70]}; {frfy[100,:,70]}; {frfy[100,100,:]} ")
print(f"z-component: {frfz[:,100,70]}; {frfz[100,:,70]}; {frfz[100,100,:]} ")
print("\nAnalytical calculation results:")
print(frf2[:,100,70])
print(frf2[100,:,70])
print(frf2[100,100,:])
print(frf2[100,100,70]); # check local minimum

Numerical calculation results:
x-component: [-0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0.
 -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0.
 -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0.
 -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0.
 -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0.
 -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0.
 -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0.
 -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0.
 -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0.
 -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0.
 -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0.
 -0. -0. -0.]; [-0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0.
 -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. 